# SQL reference 12: query plans and performance

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql/sql-reference-12-query-plans-and-performance.ipynb)

Reading `EXPLAIN` output and the physical operators, partition pruning, filter pushdown and column pruning, file layout and statistics, join strategies, adaptive execution and skew, the cost of distinct counts and set operations, repartitioning hints, and a tuning checklist. Practise it in [SQL practice 12: query plans and performance](sql-practice-12-query-plans-and-performance.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0 (Spark SQL).*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

A local Spark session, the shared tables as temporary views: `customers`, `orders`, `order_items`, `products`, `events` and `employees` (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry, two pairs of employees tie on salary, Dana out-earns her manager and Jo has no department), and three helpers: `show(query)` runs a query and prints its rows untruncated, `plan(query)` returns what `EXPLAIN` prints, and `expected_error()` prints the error class of an example that fails on purpose. This notebook also writes Parquet copies of the data: `orders_pq` (partitioned by `order_date`), `events_pq` and `items_pq`, and defines `plan_lines(query, *keywords)`, which filters the `EXPLAIN FORMATTED` output.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
# Shared sample tables as temporary views. Each statement is ordinary Spark SQL.
TABLES = {
    "customers": """CREATE OR REPLACE TEMP VIEW customers AS SELECT * FROM VALUES
  (1, 'Acme Corp',      'North', DATE'2024-01-03'),
  (2, 'Bright Retail',  'North', DATE'2024-02-11'),
  (3, 'Cyan Traders',   'South', DATE'2024-03-15'),
  (4, 'Delta Grocers',  NULL,    DATE'2024-05-02'),   -- region never arrived from the CRM
  (5, 'Echo Supplies',  'West',  DATE'2024-06-20'),
  (6, 'Foxtrot Foods',  'West',  DATE'2024-08-01'),
  (7, 'Gamma Sports',   'East',  DATE'2024-09-14'),   -- 7 and 8 have never ordered
  (8, 'Hotel Homeware', 'East',  DATE'2024-11-05')
AS customers(customer_id, name, region, signup_date)""",
    "orders": """CREATE OR REPLACE TEMP VIEW orders AS SELECT order_id, customer_id, status, order_ts, CAST(amount AS DECIMAL(12,2)) AS amount FROM VALUES
  (1,  1, 'paid',     TIMESTAMP'2026-01-05 09:15:00', 120.00),
  (2,  1, 'shipped',  TIMESTAMP'2026-01-05 10:05:00', 84.50),
  (3,  1, 'returned', TIMESTAMP'2026-01-05 11:40:00', 30.00),
  (4,  2, 'paid',     TIMESTAMP'2026-01-06 15:20:00', 260.00),
  (5,  2, 'placed',   TIMESTAMP'2026-01-07 08:05:00', 55.25),
  (6,  3, 'shipped',  TIMESTAMP'2026-01-07 17:45:00', 149.00),
  (7,  3, 'paid',     TIMESTAMP'2026-01-09 12:10:00', 310.00),
  (8,  4, 'returned', TIMESTAMP'2026-01-10 19:30:00', 89.90),
  (9,  4, 'shipped',  TIMESTAMP'2026-01-12 08:55:00', 1180.00),
  (10, 5, 'paid',     TIMESTAMP'2026-01-12 13:00:00', 74.00),
  (11, 6, 'placed',   TIMESTAMP'2026-01-13 10:25:00', 210.00),
  (12, 6, 'shipped',  TIMESTAMP'2026-01-14 00:30:00', 96.40)   -- just after midnight UTC
AS orders(order_id, customer_id, status, order_ts, amount)""",
    "order_items": """CREATE OR REPLACE TEMP VIEW order_items AS SELECT order_id, sku, qty, CAST(unit_price AS DECIMAL(8,2)) AS unit_price FROM VALUES
  (1, 'SKU-4001', 2, 12.50), (1, 'SKU-4007', 1, 95.00), (2, 'SKU-4002', 1, 34.50), (2, 'SKU-4009', 2, 25.00),
  (3, 'SKU-4002', 1, 30.00), (4, 'SKU-4003', 1, 149.00), (4, 'SKU-4004', 2, 55.50), (5, 'SKU-4005', 1, 55.25),
  (6, 'SKU-4003', 1, 149.00), (7, 'SKU-4006', 1, 189.00), (7, 'SKU-4008', 1, 121.00), (8, 'SKU-4010', 1, 89.90),
  (9, 'SKU-4011', 20, 59.00), (10, 'SKU-4012', 2, 37.00), (11, 'SKU-4013', 1, 210.00),
  (12, 'SKU-4014', 1, 41.40), (12, 'SKU-4005', 1, 55.00)
AS order_items(order_id, sku, qty, unit_price)""",
    "products": """CREATE OR REPLACE TEMP VIEW products AS SELECT sku, category, CAST(list_price AS DECIMAL(8,2)) AS list_price FROM VALUES
  ('SKU-4001', 'Kitchen', 12.50), ('SKU-4002', 'Kitchen', 34.50), ('SKU-4003', 'Garden', 149.00),
  ('SKU-4004', 'Garden', 55.50), ('SKU-4005', 'Pantry', 55.25), ('SKU-4006', 'Outdoor', 189.00),
  ('SKU-4007', 'Kitchen', 95.00), ('SKU-4008', 'Outdoor', 121.00), ('SKU-4009', 'Pantry', 25.00),
  ('SKU-4010', 'Garden', 89.90), ('SKU-4011', 'Pantry', 59.00), ('SKU-4012', 'Kitchen', 37.00),
  ('SKU-4013', 'Outdoor', 210.00)   -- SKU-4014 has no product row
AS products(sku, category, list_price)""",
    "events": """CREATE OR REPLACE TEMP VIEW events AS SELECT * FROM VALUES
  (1, 'view', TIMESTAMP'2026-01-06 09:00:00'), (1, 'add_to_cart', TIMESTAMP'2026-01-06 09:02:00'),
  (1, 'purchase', TIMESTAMP'2026-01-06 09:10:00'), (2, 'view', TIMESTAMP'2026-01-07 10:00:00'),
  (2, 'purchase', TIMESTAMP'2026-01-07 10:40:00'), (3, 'view', TIMESTAMP'2026-01-06 11:15:00'),
  (3, 'view', TIMESTAMP'2026-01-08 11:20:00'), (4, 'add_to_cart', TIMESTAMP'2026-01-09 20:05:00'),
  (4, 'purchase', TIMESTAMP'2026-01-10 08:30:00'), (5, 'view', TIMESTAMP'2026-01-11 14:45:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00'), (5, 'purchase', TIMESTAMP'2026-01-12 12:30:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00')   -- an SDK retry committed this event twice
AS events(user_id, event, ts)""",
    "employees": """CREATE OR REPLACE TEMP VIEW employees AS SELECT * FROM VALUES
  (1,  'Ada',    'Engineering', CAST(NULL AS INT), 185000, DATE'2018-03-01'),
  (2,  'Brett',  'Engineering', 1,    150000, DATE'2019-07-15'),
  (3,  'Chen',   'Engineering', 1,    150000, DATE'2020-01-06'),
  (4,  'Dana',   'Engineering', 2,    155000, DATE'2022-09-12'),   -- out-earns her manager
  (5,  'Eli',    'Sales',       1,    130000, DATE'2019-02-01'),
  (6,  'Farah',  'Sales',       5,    95000,  DATE'2021-05-17'),
  (7,  'Gus',    'Sales',       5,    95000,  DATE'2023-01-09'),
  (8,  'Hana',   'Finance',     1,    140000, DATE'2017-11-20'),
  (9,  'Ivan',   'Finance',     8,    88000,  DATE'2024-04-01'),
  (10, 'Jo',     NULL,          8,    70000,  DATE'2025-06-30')   -- department not assigned yet
AS employees(emp_id, name, dept, manager_id, salary, hire_date)""",
}
for statement in TABLES.values():
    spark.sql(statement)

import contextlib
import io
import logging
import re


def show(query, n=50, **args):
    """Run a query and print up to n rows without truncating values. Keyword arguments bind :name parameters."""
    spark.sql(query, args=args or None).show(n, truncate=False)


def plan(query, mode="formatted"):
    """The plan EXPLAIN prints for a query, as text."""
    return spark.sql(f"EXPLAIN {mode.upper()} {query}").first()[0]


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        lines = [line for line in str(error).splitlines() if line.strip()]
        message = next((line for line in lines if re.search(r"\[[A-Z][A-Z0-9_.]+\]", line)), lines[0] if lines else "")
        print(type(error).__name__ + ":", message.strip().split(". ")[0][:120])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

# File-backed copies of the sample data, so plans show scans, pruning and pushdown.
for table, ddl in {
    "orders_pq": "USING parquet PARTITIONED BY (order_date) AS SELECT order_id, customer_id, status, order_ts, amount, to_date(order_ts) AS order_date FROM orders",
    "events_pq": "USING parquet AS SELECT * FROM events",
    "items_pq": "USING parquet AS SELECT * FROM order_items",
}.items():
    spark.sql(f"DROP TABLE IF EXISTS {table}")
    spark.sql(f"CREATE TABLE {table} {ddl}")


def plan_lines(query, *keywords):
    """The lines of EXPLAIN FORMATTED that contain any of the keywords."""
    return [line.strip() for line in plan(query).splitlines() if any(k in line for k in keywords)]

## Contents

| section | entries |
| --- | --- |
| [Reading plans](#reading-plans) | [`EXPLAIN`](#explain), [Physical operators](#physical-operators) |
| [Reading less data](#reading-less-data) | [Partition pruning, filter pushdown and column pruning](#partition-pruning-filter-pushdown-and-column-pruning), [File layout and statistics](#file-layout-and-statistics) |
| [Joins and shuffles](#joins-and-shuffles) | [Join strategies](#join-strategies), [Aggregations, distinct counts and set operations](#aggregations-distinct-counts-and-set-operations), [Shuffle partitions and repartitioning hints](#shuffle-partitions-and-repartitioning-hints) |
| [Checklist](#checklist) | [Performance tuning checklist](#performance-tuning-checklist) |

## Reading plans

### `EXPLAIN`

```sql
EXPLAIN [EXTENDED | FORMATTED | COST | CODEGEN] query
```

| mode | shows |
| --- | --- |
| (default) | the physical plan as a tree |
| `EXTENDED` | parsed, analysed and optimised logical plans, then the physical plan |
| `FORMATTED` | a compact tree plus one detail block per operator (filters, read schema, join keys): the most readable |
| `COST` | the optimised logical plan with size and row-count estimates |

Read a physical plan **bottom-up**: scans at the leaves, then filters and projections, exchanges (shuffles),
aggregations and joins, and the final projection at the top. With adaptive query execution (AQE, on by default) the
plan is wrapped in `AdaptiveSparkPlan isFinalPlan=false`: the real plan is decided at run time and visible in the
Spark UI or after execution.

**Notes:** Postgres has `EXPLAIN ANALYZE` (runs the query and reports actual rows and times); Snowflake and BigQuery
show executed plans in their query profile UIs.

In [ ]:
print(plan("""
SELECT c.region, sum(o.amount) AS revenue
FROM orders_pq o JOIN customers c ON c.customer_id = o.customer_id
WHERE o.status <> 'returned'
GROUP BY c.region
"""))

### Physical operators

| operator | meaning | watch for |
| --- | --- | --- |
| `Scan parquet` / `FileScan` | read files | `PartitionFilters`, `PushedFilters`, `ReadSchema` (below) |
| `LocalTableScan` | in-memory literal data (views over `VALUES`) | |
| `Filter`, `Project` | row filter, column selection and expressions | |
| `Exchange hashpartitioning(k, n)` | a **shuffle**: rows move across the network by key | the main cost of joins, aggregations and windows |
| `Exchange SinglePartition` | everything to one task | global sorts or windows without `PARTITION BY` |
| `HashAggregate` (partial, then final) | two-phase aggregation around a shuffle | `Expand` underneath means several `DISTINCT` aggregates |
| `BroadcastHashJoin` | small side copied to every task, no shuffle of the large side | the cheapest equi-join |
| `SortMergeJoin` | both sides shuffled and sorted on the key | standard for two large inputs |
| `BroadcastNestedLoopJoin`, `CartesianProduct` | no equality condition | dangerous unless one side is tiny |
| `Window` | window functions, after an exchange on the `PARTITION BY` keys | |
| `TakeOrderedAndProject` | `ORDER BY ... LIMIT n` done as a top-n per task | far cheaper than a full sort |

In [ ]:
for line in plan_lines("SELECT * FROM orders_pq ORDER BY amount DESC LIMIT 3", "TakeOrderedAndProject", "Sort"):
    print(line)
for line in plan_lines("SELECT customer_id, sum(amount) OVER (ORDER BY order_ts) FROM orders_pq", "Exchange", "Window"):
    print(line)

## Reading less data

### Partition pruning, filter pushdown and column pruning

| optimisation | in the plan | how to get it |
| --- | --- | --- |
| **partition pruning** | `PartitionFilters: [...]` on the scan | filter on the partition column; Spark evaluates even expressions over partition values, so whole directories are skipped |
| **filter pushdown** | `PushedFilters: [...]` | simple comparisons on columns (`=`, `<`, `IN`, `IS NULL`); the file format skips row groups using min/max statistics |
| **column pruning** | `ReadSchema: struct<...>` lists only used columns | select only the columns you need; columnar formats read nothing else |

**What blocks pushdown:** wrapping a data column in most functions, casts or arithmetic
(`date_format(ts, 'yyyy-MM-dd') = ...`, `hour(ts) = 9`, `CAST(id AS STRING) = '7'`, `id + 1 = 6`), UDFs, and `OR`
across different columns. Spark's optimiser rewrites a few common forms for you (`to_date(ts) = DATE'...'` becomes a
range), but do not rely on it: write a range on the raw column (a **sargable** predicate),
`ts >= TIMESTAMP'2026-01-12' AND ts < TIMESTAMP'2026-01-13'`, which every engine can use.

In [ ]:
print(plan_lines("SELECT order_id FROM orders_pq WHERE order_date >= DATE'2026-01-12'", "PartitionFilters", "ReadSchema"))
print(plan_lines("SELECT user_id FROM events_pq WHERE date_format(ts, 'yyyy-MM-dd') = '2026-01-12'", "PushedFilters"))   # only IsNotNull is pushed
print(plan_lines("SELECT user_id FROM events_pq WHERE ts >= TIMESTAMP'2026-01-12 00:00:00' AND ts < TIMESTAMP'2026-01-13 00:00:00'", "PushedFilters"))

### File layout and statistics

| technique | effect |
| --- | --- |
| right-sized files (128 MB to 1 GB) | many small files make scans slow (per-file overhead) and planning expensive; compact them (`OPTIMIZE` on Delta) |
| clustering (`ZORDER BY`, `CLUSTER BY` on Delta) | co-locates similar values so min/max statistics skip more files for selective filters |
| partitioning | only by low-cardinality columns that most queries filter on |
| `ANALYZE TABLE t COMPUTE STATISTICS [FOR COLUMNS a, b]` | table and column statistics for the cost-based optimiser (join order, broadcast decisions) |
| `CACHE TABLE` | keep a hot, reused intermediate result in memory |

**Notes:** OLTP databases rely on B-tree **indexes** and covering indexes; columnar warehouses and lakehouses rely on
partitioning, clustering and zone maps (min/max per block) instead. Sargable predicates matter for both.

In [ ]:
spark.sql("ANALYZE TABLE orders_pq COMPUTE STATISTICS FOR COLUMNS customer_id, amount")
spark.sql("DESCRIBE TABLE EXTENDED orders_pq").where("col_name = 'Statistics'").show(truncate=False)

## Joins and shuffles

### Join strategies

| strategy | chosen when | cost |
| --- | --- | --- |
| broadcast hash join | one side is estimated below `spark.sql.autoBroadcastJoinThreshold` (10 MB), or hinted `/*+ BROADCAST(t) */` | no shuffle of the large side |
| shuffle hash join | hinted `/*+ SHUFFLE_HASH(t) */`, or preferred by AQE when partitions fit in memory | shuffle both sides, no sort |
| sort-merge join | the default for two large sides (`/*+ MERGE(t) */`) | shuffle and sort both sides |
| broadcast nested loop | no equality in the join condition | compares every pair: keep one side tiny |

**Adaptive query execution** re-plans at run time from real sizes: it turns a sort-merge join into a broadcast join
when one side turns out small, coalesces small shuffle partitions, and splits **skewed** partitions
(`spark.sql.adaptive.skewJoin.enabled`). When skew remains (one key with most of the rows), **salt** the key: add a
random suffix on the big side and replicate the small side once per suffix.

In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")      # pretend every table is large
print(plan_lines("SELECT o.order_id, c.name FROM orders_pq o JOIN customers c USING (customer_id)", "Join "))
print(plan_lines("SELECT /*+ BROADCAST(c) */ o.order_id, c.name FROM orders_pq o JOIN customers c USING (customer_id)", "Join "))
print(plan_lines("SELECT * FROM customers c WHERE EXISTS (SELECT 1 FROM orders_pq o WHERE o.customer_id = c.customer_id)", "Join "))
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760")

### Aggregations, distinct counts and set operations

| pattern | plan | cheaper alternative |
| --- | --- | --- |
| `GROUP BY` | partial `HashAggregate`, shuffle, final `HashAggregate` | filter and project before grouping |
| several `count(DISTINCT ...)` | an `Expand` that multiplies rows by the number of distinct aggregates | `approx_count_distinct` (HyperLogLog), or one distinct per query |
| `UNION` | `Union` plus a de-duplicating `HashAggregate` (shuffle) | `UNION ALL` when duplicates are impossible or wanted |
| `SELECT DISTINCT` to fix join duplicates | an extra shuffle hiding a fan-out bug | fix the join grain |
| a window without `PARTITION BY` | `Exchange SinglePartition`: one task does everything | partition the window |

In [ ]:
print(plan_lines("SELECT count(DISTINCT customer_id), count(DISTINCT status) FROM orders_pq", "Expand"))
print(plan_lines("SELECT approx_count_distinct(customer_id) FROM orders_pq", "Expand", "approx_count_distinct(customer_id#")[:1])
print(len(plan_lines("SELECT customer_id FROM orders_pq UNION SELECT customer_id FROM customers", "HashAggregate")) > 0,
      len(plan_lines("SELECT customer_id FROM orders_pq UNION ALL SELECT customer_id FROM customers", "HashAggregate")) > 0)

### Shuffle partitions and repartitioning hints

| setting or hint | does |
| --- | --- |
| `spark.sql.shuffle.partitions` | partitions after a shuffle (default 200; AQE coalesces small ones) |
| `/*+ REPARTITION(n, col) */` | shuffle into `n` partitions by `col` before writing |
| `/*+ COALESCE(n) */` | merge into fewer partitions without a full shuffle (fewer output files) |
| `/*+ REBALANCE(col) */` | evenly sized output partitions, splitting skew (AQE) |

In [ ]:
print(plan_lines("SELECT /*+ REPARTITION(4, customer_id) */ * FROM orders_pq", "Exchange"))

## Checklist

### Performance tuning checklist

1. **Read less:** filter on partition columns, write sargable predicates, select only needed columns.
2. **Shuffle less:** broadcast small dimensions, pre-aggregate before joining, avoid `DISTINCT` and `UNION` you do not
   need, partition windows.
3. **Fix the data layout:** compact small files, cluster by common filter columns, keep statistics current.
4. **Handle skew:** let AQE split skewed partitions; salt keys that remain hot.
5. **Prefer set-based SQL:** window functions over self-joins, `EXISTS` over `IN` + `DISTINCT`, built-in functions over
   UDFs.
6. **Measure:** compare plans and run times before and after; check the Spark UI for spill, skewed task times and
   shuffle sizes.

**Next:** practise these entries in [SQL practice 12: query plans and performance](sql-practice-12-query-plans-and-performance.ipynb).